In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from tpot import TPOTClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics import PrecisionRecallDisplay

import utils

# Read data

In [ ]:
df = pd.read_parquet('data-input/flirt-wesad-acc-bvp-eda-temp-60-10.parquet')

In [ ]:
# remove columns (see EDA)
columns_to_drop = ['eda_EDA_n_sign_changes',
 'temp_TEMP_peaks',
 'acc_y_entropy',
 'acc_l2_n_sign_changes',
 'acc_x_entropy',
 'acc_z_entropy',
 'temp_l2_n_sign_changes',
 'bvp_BVP_entropy',
 'temp_TEMP_n_sign_changes',
 'temp_l2_peaks',
 'eda_l2_n_sign_changes']

df = df.drop(columns=columns_to_drop)

In [ ]:
# split into train and test
df_train, df_test = utils.create_train_test(df, 5, 'subject', 'label')

X_train, y_train, groups_train = utils.split_df(df_train, 'subject', 'label')
X_test, y_test, groups_test = utils.split_df(df_test, 'subject', 'label')

# remove correlated features from train
X_train, selected_features = utils.remove_correlated_features(X_train, 0.8)

# remove the same columns from test
X_test = X_test[selected_features]

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
X_train.columns

In [ ]:
# Check train and test set sizes
print('Percentage train set:', len(y_train)/(len(y_train)+len(y_test)))
print('Percentage test set:', len(y_test)/(len(y_train)+len(y_test)))

print('\nClass distribution in train set: \n', y_train['label'].value_counts(normalize=True), '\n')

print('Class distribution in test set: \n', y_test['label'].value_counts(normalize=True), '\n')

# TPOT

In [ ]:
tpot = TPOTClassifier(generations=5,
                      population_size=5,
                      scoring='f1',
                      cv=3,
                      n_jobs=-1,
                      verbosity=3,
                      random_state=0)

In [ ]:

%%time
tpot.fit(X_train, y_train.values.ravel(), groups=groups_train.values.ravel())

In [ ]:
print(f"TPOT score (F1) on test data: {tpot.score(X_test, y_test.values.ravel()):.2f}")

In [ ]:
y_test_predict = tpot.predict(X_test)
y_test_predict_proba = tpot.predict_proba(X_test)[:,1]

In [ ]:
print('accuracy: '+str(accuracy_score(y_test, y_test_predict)))
print('precision: '+str(precision_score(y_test, y_test_predict)))
print('recall: '+str(recall_score(y_test, y_test_predict)))
print('f1_score: '+str(f1_score(y_test, y_test_predict)))
print('roc_auc: '+str(roc_auc_score(y_test, y_test_predict)))
print('Class distribution in test set: \n', y_test['label'].value_counts(normalize=True), '\n')
print('average_precision: '+str(average_precision_score(y_test, y_test_predict_proba)))

In [ ]:
confusion_matrix(y_test, y_test_predict)

In [ ]:
prd = PrecisionRecallDisplay.from_predictions(y_test, y_test_predict_proba, name='Model')
_ = prd.ax_.set_title('2-class Precision-Recall curve')
print('The baseline to beat is the percentage of positive cases:', y_test['label'].value_counts(normalize=True)[1])

## Interpretation
I created a TPOT baseline model in this notebook.

The data used was created with FLIRT with a window_size of 60 and a step size of 10.

The data of each user is either in the train or in the test set. Internally, TPOT does a cross-validation with the training data, and again only uses the data of each user in either training or validation set.

Hyperparameters, i.e., aspects that could be changed in future iterations:

Calculating the features with another library instead of FLIRT.
Using different window_sizes and step sizes.
Test if we can predict stress by only using a subset of the available sensors.
The results so far are very promising: TPOT returned a pipeline with a performance on the test set of F1=0.88.